the dataset LIS-DS is composed of different scenario, we propose to focus here on Bruteforce_CWE-307, CVE-2014-0160, CWE-89-SQL-injection. The full dataset is available here : https://drive.proton.me/urls/BWKRGQK994#fCK9JKL93Sjm

In [31]:
from pathlib import Path
# declare the dataset path you want to use 
path_data = "/Users/tristan/Documents/dev/01_Cours_CS/08_MLNS/intrusion-detection/data/LIS-DS/CWE-89-SQL-injection"

In [29]:
import zipfile
import json
import re
import pandas as pd
import io
from collections import Counter

import numpy as np
import pandas as pd
import networkx as nx

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

In [27]:
zip_path = Path("/Users/tristan/Documents/dev/01_Cours_CS/08_MLNS/intrusion-detection/data/LIS-DS/CWE-89-SQL-injection.zip")
extract_dir = Path("/Users/tristan/Documents/dev/01_Cours_CS/08_MLNS/intrusion-detection/data/LIS-DS/CWE-89-SQL-injection")

extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

print("Extracted to:", extract_dir)
print("Top-level contents:")
for p in extract_dir.iterdir():
    print("-", p.name)

KeyboardInterrupt: 

In [ ]:
import zipfile
from pathlib import Path

def explore_zip(zip_path, max_depth=2, indent=0):
    """
    pour explorer un zip 
    """
    prefix = "  " * indent
    print(f"{prefix} - {zip_path.name}")
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            file_list = z.namelist()

            for name in file_list[:20]:  # limite affichage
                print(f"{prefix}  - {name}")

            if len(file_list) > 20:
                print(f"{prefix}  ... ({len(file_list)} files total)")

            if max_depth > 0:
                for name in file_list:
                    if name.endswith(".zip"):
                        print(f"{prefix}  🔽 Found nested zip: {name}")

    except zipfile.BadZipFile:
        print(f"{prefix} not a valid zip")


def explore_folder(folder_path, max_zips=5):
    folder = Path(folder_path)
    zip_files = list(folder.rglob("*.zip"))
    print(f"Found {len(zip_files)} zip files")
    for zip_path in zip_files[:max_zips]:
        explore_zip(zip_path)
        print("-" * 50)

explore_folder("/Users/tristan/Documents/dev/01_Cours_CS/08_MLNS/intrusion-detection/data/LIS-DS/Bruteforce_CWE-307/Bruteforce_CWE-307", max_zips=5)

Found 1177 zip files
 - colossal_napier_5719.zip
  - colossal_napier_5719.json
  - colossal_napier_5719.sc
  - colossal_napier_5719.pcap
  - colossal_napier_5719.res
--------------------------------------------------
 - slow_taussig_7121.zip
  - slow_taussig_7121.json
  - slow_taussig_7121.sc
  - slow_taussig_7121.pcap
  - slow_taussig_7121.res
--------------------------------------------------
 - fat_lederberg_9674.zip
  - fat_lederberg_9674.json
  - fat_lederberg_9674.sc
  - fat_lederberg_9674.pcap
  - fat_lederberg_9674.res
--------------------------------------------------
 - sticky_liskov_8759.zip
  - sticky_liskov_8759.json
  - sticky_liskov_8759.sc
  - sticky_liskov_8759.pcap
  - sticky_liskov_8759.res
--------------------------------------------------
 - blue_haibt_8901.zip
  - blue_haibt_8901.json
  - blue_haibt_8901.sc
  - blue_haibt_8901.pcap
  - blue_haibt_8901.res
--------------------------------------------------


Let's preview what a .sc looks like... 

In [ ]:
def inspect_one_sample(zip_path, max_sc_lines=20):
    zip_path = Path(zip_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()
        print("ZIP:", zip_path.name)
        print("Contents:")
        for n in names:
            print(" -", n)

        sc_files = [n for n in names if n.endswith(".sc")]
        json_files = [n for n in names if n.endswith(".json")]
        res_files = [n for n in names if n.endswith(".res")]

        print("\n - JSON preview - ")
        if json_files:
            with z.open(json_files[0]) as f:
                content = f.read().decode("utf-8", errors="ignore")
                try:
                    obj = json.loads(content)
                    print(json.dumps(obj, indent=2)[:4000])
                except Exception:
                    print(content[:4000])
        else:
            print("No .json file found")

        print("\n - SC preview -")
        if sc_files:
            with z.open(sc_files[0]) as f:
                text = f.read().decode("utf-8", errors="ignore")
                lines = text.splitlines()
                for line in lines[:max_sc_lines]:
                    print(line)
                print(f"\nTotal .sc lines: {len(lines)}")
        else:
            print("No .sc file found")

        print("\n- RES preview -")
        if res_files:
            with z.open(res_files[0]) as f:
                text = f.read().decode("utf-8", errors="ignore")
                print(text[:2000])
        else:
            print("No .res file found")

put your path to the dataset

In [33]:
inspect_one_sample("/Users/tristan/Documents/dev/01_Cours_CS/08_MLNS/intrusion-detection/data/LIS-DS/CWE-89-SQL-injection/CWE-89-SQL-injection/test/normal_and_attack/bald_mclean_5384.zip")

ZIP: bald_mclean_5384.zip
Contents:
 - bald_mclean_5384.json
 - bald_mclean_5384.sc
 - bald_mclean_5384.pcap
 - bald_mclean_5384.res

 - JSON preview - 
{
  "container": [
    {
      "ip": "192.168.48.2",
      "name": "b213e5059efcf1e9",
      "role": "attacker"
    },
    {
      "ip": "192.168.48.3",
      "name": "206787206e0c8f9e",
      "role": "victim"
    }
  ],
  "exploit": true,
  "exploit_name": "default",
  "image": "victim_sql",
  "recording_time": 36,
  "time": {
    "container_ready": {
      "absolute": 1631084375.517234,
      "source": "CONTROL_SCRIPT"
    },
    "exploit": [
      {
        "absolute": 1631084392.965272,
        "name": "attack",
        "source": "SYSDIG"
      }
    ],
    "warmup_end": {
      "absolute": 1631084378.519487,
      "source": "CONTROL_SCRIPT"
    }
  }
}

 - SC preview -
1631084378899769357 101 4017297 mysqld 4017347 futex < res=-110(ETIMEDOUT) 
1631084378899772486 101 4017297 mysqld 4017347 futex > addr=7FD52E468240 op=129(FUTEX_PR

In [34]:
SC_LINE_RE = re.compile(
    r"^(?P<timestamp>\d+)\s+"
    r"(?P<field2>\S+)\s+"
    r"(?P<field3>\S+)\s+"
    r"(?P<procname>\S+)\s+"
    r"(?P<field5>\S+)\s+"
    r"(?P<syscall>\S+)\s+"
    r"(?P<direction>[<>])\s*"
    r"(?P<rest>.*)$"
)

RES_RE = re.compile(r"\bres=(-?\d+)\b")


def parse_sc_line(line):
    line = line.strip()
    m = SC_LINE_RE.match(line)
    if not m:
        return None

    d = m.groupdict()
    rest = d["rest"]

    res_match = RES_RE.search(rest)
    res_value = int(res_match.group(1)) if res_match else None

    return {
        "timestamp": int(d["timestamp"]),
        "procname": d["procname"],
        "syscall": d["syscall"],
        "direction": d["direction"],
        "raw_rest": rest,
        "res": res_value,
    }


def load_lidds_sample(zip_path):
    zip_path = Path(zip_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        json_name = next(n for n in names if n.endswith(".json"))
        sc_name = next(n for n in names if n.endswith(".sc"))

        with z.open(json_name) as f:
            meta = json.loads(f.read().decode("utf-8", errors="ignore"))

        with z.open(sc_name) as f:
            lines = f.read().decode("utf-8", errors="ignore").splitlines()

    events = [parse_sc_line(line) for line in lines]
    events = [e for e in events if e is not None]

    df = pd.DataFrame(events)

    label = int(bool(meta.get("exploit", False)))

    return {
        "file": str(zip_path),
        "label": label,
        "meta": meta,
        "events_df": df,
    }

In [ ]:
sample = load_lidds_sample("/Users/tristan/Documents/dev/01_Cours_CS/08_MLNS/intrusion-detection/data/LIS-DS/Bruteforce_CWE-307/Bruteforce_CWE-307/training/abundant_dhawan_2607.zip")

print("Label:", sample["label"])
print(sample["events_df"].head())
print(sample["events_df"]["direction"].value_counts())
print(sample["events_df"]["syscall"].value_counts().head(20))

Label: 0
             timestamp procname     syscall direction raw_rest  res
0  1631011407593552353  apache2      select         <    res=0  0.0
1  1631011407593556052  apache2       wait4         >           NaN
2  1631011407593558216  apache2       wait4         <           NaN
3  1631011407593559018  apache2      select         >           NaN
4  1631011407697748500  apache2  epoll_wait         <    res=1  1.0
direction
>    11469
<    11468
Name: count, dtype: int64
syscall
read           2932
open           2104
close          1745
fstat          1702
umask          1674
writev         1430
poll           1399
setitimer      1340
write           798
fcntl           716
semop           715
brk             712
getcwd          672
chdir           670
accept          358
getsockname     358
stat            358
shutdown        358
epoll_wait      357
times           356
Name: count, dtype: int64


we then must convert our data to a sequence of system call

In [ ]:
def df_to_syscall_sequence(events_df, direction_filter=">"):
    df = events_df.copy()

    if direction_filter is not None:
        df = df[df["direction"] == direction_filter]

    return df["syscall"].tolist()

In [ ]:
seq = df_to_syscall_sequence(sample["events_df"], direction_filter=">")
print(seq[:30])
print("Length:", len(seq))

['wait4', 'select', 'accept', 'semop', 'getsockname', 'epoll_wait', 'open', 'read', 'close', 'fcntl', 'fcntl', 'read', 'writev', 'poll', 'read', 'writev', 'poll', 'read', 'stat', 'open', 'open', 'open', 'fstat', 'read', 'close', 'setitimer', 'rt_sigaction', 'rt_sigprocmask', 'poll', 'read']
Length: 11469


In [ ]:
# Dataloader, according to the above inspection

class LIDDSLoader:
    """
    Expected structure:
        root/
            scenario_name/
                training/ -> not used, came from a preprocessed repo
                    *.zip
                validation/ -> not used, came from a preprocessed repo
                    *.zip
                test/
                    *.zip
    """

    SC_LINE_RE = re.compile(
        r"^(?P<timestamp>\d+)\s+"
        r"(?P<field2>\S+)\s+"
        r"(?P<field3>\S+)\s+"
        r"(?P<procname>\S+)\s+"
        r"(?P<field5>\S+)\s+"
        r"(?P<syscall>\S+)\s+"
        r"(?P<direction>[<>])\s*"
        r"(?P<rest>.*)$"
    )

    RES_RE = re.compile(r"\bres=(-?\d+)\b")

    def __init__(self, root_dir, direction_filter=">", keep_only_successful_parsed_lines=True):
        self.root = Path(root_dir)
        self.direction_filter = direction_filter
        self.keep_only_successful_parsed_lines = keep_only_successful_parsed_lines

    def _collect_zip_files(self):
        return sorted(self.root.rglob("*.zip"))

    def _infer_split_and_scenario(self, zip_path: Path):
        parts_lower = [p.lower() for p in zip_path.parts]

        split = None
        split_aliases = {
            "train": "train",
            "training": "train",
            "validation": "validation",
            "val": "validation",
            "test": "test",
            "testing": "test",
        }

        split_idx = None
        for i, part in enumerate(parts_lower):
            if part in split_aliases:
                split = split_aliases[part]
                split_idx = i
                break

        scenario = "unknown"
        if split_idx is not None and split_idx > 0:
            scenario = zip_path.parts[split_idx - 1]

        return split, scenario

    def _parse_sc_line(self, line: str):
        line = line.strip()
        if not line:
            return None

        m = self.SC_LINE_RE.match(line)
        if not m:
            return None

        d = m.groupdict()
        rest = d["rest"]

        res_match = self.RES_RE.search(rest)
        res_value = int(res_match.group(1)) if res_match else np.nan

        return {
            "timestamp": int(d["timestamp"]),
            "procname": d["procname"],
            "syscall": d["syscall"],
            "direction": d["direction"],
            "res": res_value,
            "raw_rest": rest,
        }

    def _load_sample_from_zip(self, zip_path: Path):
        with zipfile.ZipFile(zip_path, "r") as z:
            names = z.namelist()

            json_files = [n for n in names if n.endswith(".json")]
            sc_files = [n for n in names if n.endswith(".sc")]

            if len(json_files) == 0:
                raise ValueError(f"No .json found in {zip_path}")
            if len(sc_files) == 0:
                raise ValueError(f"No .sc found in {zip_path}")

            json_name = json_files[0]
            sc_name = sc_files[0]

            with z.open(json_name) as f:
                meta = json.loads(f.read().decode("utf-8", errors="ignore"))

            with z.open(sc_name) as f:
                sc_text = f.read().decode("utf-8", errors="ignore")

        lines = sc_text.splitlines()
        events = [self._parse_sc_line(line) for line in lines]

        if self.keep_only_successful_parsed_lines:
            events = [e for e in events if e is not None]

        events_df = pd.DataFrame(events)

        if len(events_df) == 0:
            raise ValueError(f"No parsed syscall lines in {zip_path}")

        if self.direction_filter is not None:
            events_df = events_df[events_df["direction"] == self.direction_filter].copy()

        if len(events_df) == 0:
            raise ValueError(f"No events left after direction filtering in {zip_path}")

        events_df = events_df.sort_values("timestamp").reset_index(drop=True)

        label = int(bool(meta.get("exploit", False)))

        return meta, events_df, label

    def load_all(self):
        rows = []

        for zip_path in self._collect_zip_files():
            split, scenario = self._infer_split_and_scenario(zip_path)

            if split is None:
                continue

            try:
                meta, events_df, label = self._load_sample_from_zip(zip_path)
            except Exception as e:
                print(f"[WARN] skipping {zip_path}: {e}")
                continue

            rows.append({
                "file": str(zip_path),
                "label": label,
                "scenario": scenario,
                "split": split,
                "meta": meta,
                "trace_df": events_df,
                "sequence": events_df["syscall"].tolist(),
            })

        return rows


# construction of the graph 

def trace_to_graph(trace_df: pd.DataFrame) -> nx.DiGraph:
    G = nx.DiGraph()

    if len(trace_df) == 0:
        return G

    total_events = len(trace_df)
    grouped = trace_df.groupby("syscall")

    for syscall, group in grouped:
        count = len(group)
        freq = count / total_events

        unique_proc = group["procname"].nunique() if "procname" in group.columns else 1

        exit_events = group[group["res"].notna()] if "res" in group.columns else group.iloc[:0]
        if len(exit_events) > 0:
            error_rate = float((exit_events["res"] < 0).mean())
            mean_res = float(exit_events["res"].mean())
            std_res = float(exit_events["res"].std()) if len(exit_events) > 1 else 0.0
        else:
            error_rate = 0.0
            mean_res = 0.0
            std_res = 0.0

        timestamps = group["timestamp"].values.astype(float)
        if len(timestamps) > 1:
            timestamps = np.sort(timestamps)
            deltas = np.diff(timestamps)
            mean_gap = float(np.mean(deltas))
            std_gap = float(np.std(deltas))
        else:
            mean_gap = 0.0
            std_gap = 0.0

        # IMPORTANT : this is where we add the features of the nodes in the Graph 
        G.add_node(
            syscall,
            count=float(count),
            freq=float(freq),
            unique_proc=float(unique_proc),
            error_rate=float(error_rate),
            mean_res=float(mean_res),
            std_res=float(std_res),
            mean_gap=float(mean_gap),
            std_gap=float(std_gap),
        )

    seq = trace_df["syscall"].tolist()
    for i in range(len(seq) - 1):
        u = seq[i]
        v = seq[i + 1]
        if G.has_edge(u, v):
            G[u][v]["weight"] += 1.0
        else:
            G.add_edge(u, v, weight=1.0)

    if G.number_of_nodes() > 0:
        pagerank = nx.pagerank(G, weight="weight") if G.number_of_edges() > 0 else {n: 0.0 for n in G.nodes()}
        for n in G.nodes():
            G.nodes[n]["in_deg"] = float(G.in_degree(n))
            G.nodes[n]["out_deg"] = float(G.out_degree(n))
            G.nodes[n]["in_wdeg"] = float(G.in_degree(n, weight="weight"))
            G.nodes[n]["out_wdeg"] = float(G.out_degree(n, weight="weight"))
            G.nodes[n]["pagerank"] = float(pagerank[n])
            G.nodes[n]["self_loop"] = float(G.has_edge(n, n))

    return G


# we build the vocabulary of the system calls

def build_syscall_vocab(rows):
    unique_syscalls = sorted({s for row in rows for s in row["sequence"]}, key=str)
    return {s: i for i, s in enumerate(unique_syscalls)}


# convert the graph to pyg in order to be used with torch-geometric

NODE_FEATURE_NAMES = [
    "freq",
    "count",
    "unique_proc",
    "error_rate",
    "mean_res",
    "std_res",
    "mean_gap",
    "std_gap",
    "in_deg",
    "out_deg",
    "in_wdeg",
    "out_wdeg",
    "pagerank",
    "self_loop",
]


def graph_to_pyg_data(G, label, syscall_vocab, file_path="", scenario="", split=""):
    nodes = list(G.nodes())
    node_to_idx = {n: i for i, n in enumerate(nodes)}

    x = []
    node_ids = []

    for node in nodes:
        attrs = G.nodes[node]
        x.append([float(attrs.get(name, 0.0)) for name in NODE_FEATURE_NAMES])
        node_ids.append(syscall_vocab[node])

    x = torch.tensor(x, dtype=torch.float)
    node_ids = torch.tensor(node_ids, dtype=torch.long)

    edges = []
    edge_weight = []

    for u, v, attrs in G.edges(data=True):
        edges.append([node_to_idx[u], node_to_idx[v]])
        edge_weight.append(float(attrs.get("weight", 1.0)))

    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_weight = torch.empty((0,), dtype=torch.float)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_weight = torch.tensor(edge_weight, dtype=torch.float)

    data = Data(
        x=x,
        node_ids=node_ids,
        edge_index=edge_index,
        edge_weight=edge_weight,
        y=torch.tensor([label], dtype=torch.long),
    )
    data.file_path = file_path
    data.scenario = scenario
    data.split = split
    return data


def rows_to_dataset(rows, syscall_vocab):
    dataset = []
    for row in rows:
        G = trace_to_graph(row["trace_df"])
        data = graph_to_pyg_data(
            G,
            label=row["label"],
            syscall_vocab=syscall_vocab,
            file_path=row["file"],
            scenario=row["scenario"],
            split=row["split"],
        )
        dataset.append(data)
    return dataset


# as explained above we only keep the folder /test  

def keep_only_test_rows(rows):
    test_rows = [r for r in rows if r["split"] == "test"]
    return test_rows

def stratified_split_rows(rows, train_size=0.7, val_size=0.15, test_size=0.15, random_state=42):
    assert abs(train_size + val_size + test_size - 1.0) < 1e-8

    labels = np.array([r["label"] for r in rows])
    indices = np.arange(len(rows))

    idx_train, idx_tmp = train_test_split(
        indices,
        test_size=(1.0 - train_size),
        random_state=random_state,
        stratify=labels,
    )

    tmp_labels = labels[idx_tmp]
    val_ratio_adjusted = val_size / (val_size + test_size)

    idx_val, idx_test = train_test_split(
        idx_tmp,
        test_size=(1.0 - val_ratio_adjusted),
        random_state=random_state,
        stratify=tmp_labels,
    )

    train_rows = [rows[i] for i in idx_train]
    val_rows = [rows[i] for i in idx_val]
    test_rows = [rows[i] for i in idx_test]

    for r in train_rows:
        r["resplit"] = "train"
    for r in val_rows:
        r["resplit"] = "validation"
    for r in test_rows:
        r["resplit"] = "test"

    return train_rows, val_rows, test_rows

def print_split_stats(name, rows):
    labels = [r["label"] for r in rows]
    print(f"{name}: {Counter(labels)}")

def assert_two_classes(rows, name):
    labels = sorted(set(r["label"] for r in rows))
    print(f"{name} classes: {labels}")
    if len(labels) < 2:
        raise ValueError(f"{name} contains only one class: {labels}")

# feature scaling 

def fit_feature_scaler(train_dataset):
    X = torch.cat([d.x for d in train_dataset], dim=0).cpu().numpy()
    scaler = StandardScaler()
    scaler.fit(X)
    return scaler

def apply_feature_scaler(dataset, scaler):
    for d in dataset:
        x_scaled = scaler.transform(d.x.cpu().numpy())
        d.x = torch.tensor(x_scaled, dtype=torch.float)
    return dataset


# weight class to treat class imbalance 

def compute_class_weights(dataset):
    labels = [int(d.y.item()) for d in dataset]
    counts = Counter(labels)
    total = sum(counts.values())
    num_classes = len(counts)

    weights = []
    for cls in [0, 1]:
        if cls in counts:
            weights.append(total / (num_classes * counts[cls]))
        else:
            weights.append(1.0)

    return torch.tensor(weights, dtype=torch.float)


# model GNN class 

class LIDDSGNN(nn.Module):
    def __init__(self, num_syscalls, num_numeric_features, emb_dim=32, hidden_dim=64, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(num_syscalls, emb_dim)
        in_dim = emb_dim + num_numeric_features

        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)

        self.dropout = dropout

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, data):
        emb = self.embedding(data.node_ids)
        x = torch.cat([emb, data.x], dim=1)

        x = self.conv1(x, data.edge_index, data.edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, data.edge_index, data.edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, data.edge_index, data.edge_weight)
        x = F.relu(x)

        x = global_mean_pool(x, data.batch)
        return self.classifier(x)


# train/eval, very classical

def train_one_epoch(model, loader, optimizer, device, class_weights=None):
    model.train()
    total_loss = 0.0
    total_n = 0

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad()
        logits = model(batch)
        loss = F.cross_entropy(logits, batch.y.view(-1), weight=class_weights)
        loss.backward()
        optimizer.step()

        n = batch.num_graphs
        total_loss += loss.item() * n
        total_n += n

    return total_loss / max(total_n, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    y_true, y_pred, y_prob = [], [], []

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        probs = F.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(dim=1)

        y_true.extend(batch.y.view(-1).cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        y_prob.extend(probs.cpu().numpy().tolist())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    if len(set(y_true)) < 2:
        auc = float("nan")
    else:
        auc = roc_auc_score(y_true, y_prob)

    return {
        "acc": acc,
        "f1": f1,
        "auc": auc,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }


def fit_model(model, train_loader, val_loader, device, epochs=30, lr=1e-3, weight_decay=1e-4, class_weights=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, train_loader, optimizer, device, class_weights=class_weights)
        train_metrics = evaluate(model, train_loader, device)
        val_metrics = evaluate(model, val_loader, device)

        print(
            f"Epoch {epoch:03d} | "
            f"loss={loss:.4f} | "
            f"train_acc={train_metrics['acc']:.4f} | "
            f"train_f1={train_metrics['f1']:.4f} | "
            f"val_acc={val_metrics['acc']:.4f} | "
            f"val_f1={val_metrics['f1']:.4f} | "
            f"val_auc={val_metrics['auc']:.4f}"
        )

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


# let's put everything together 

def main():
    root_dir = path_data

    loader = LIDDSLoader(root_dir, direction_filter=">")
    rows = loader.load_all()

    print(f"Number of traces (all original splits): {len(rows)}")
    print(f"Original split counts: {Counter(r['split'] for r in rows)}")
    print(f"Original label counts: {Counter(r['label'] for r in rows)}")

    # keep only original test because it contains both classes
    rows = keep_only_test_rows(rows)
    print(f"\nUsing only original test split: {len(rows)}")
    print(f"Label counts in original test: {Counter(r['label'] for r in rows)}")

    # resplit
    train_rows, val_rows, test_rows = stratified_split_rows(
        rows,
        train_size=0.7,
        val_size=0.15,
        test_size=0.15,
        random_state=42,
    )

    print_split_stats("train", train_rows)
    print_split_stats("val", val_rows)
    print_split_stats("test", test_rows)

    assert_two_classes(train_rows, "train")
    assert_two_classes(val_rows, "val")
    assert_two_classes(test_rows, "test")

    syscall_vocab = build_syscall_vocab(rows)
    print(f"Unique syscalls: {len(syscall_vocab)}")

    train_dataset = rows_to_dataset(train_rows, syscall_vocab)
    val_dataset = rows_to_dataset(val_rows, syscall_vocab)
    test_dataset = rows_to_dataset(test_rows, syscall_vocab)

    scaler = fit_feature_scaler(train_dataset)
    train_dataset = apply_feature_scaler(train_dataset, scaler)
    val_dataset = apply_feature_scaler(val_dataset, scaler)
    test_dataset = apply_feature_scaler(test_dataset, scaler)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = LIDDSGNN(
        num_syscalls=len(syscall_vocab),
        num_numeric_features=len(NODE_FEATURE_NAMES),
        emb_dim=32,
        hidden_dim=64,
        dropout=0.3,
    ).to(device)

    class_weights = compute_class_weights(train_dataset).to(device)
    print("Class weights:", class_weights)

    model = fit_model(
        model,
        train_loader,
        val_loader,
        device,
        epochs=30,
        lr=1e-3,
        weight_decay=1e-4,
        class_weights=class_weights,
    )

    test_metrics = evaluate(model, test_loader, device)

    print("\n - TEST -")
    print(f"Accuracy: {test_metrics['acc']:.4f}")
    print(f"F1:       {test_metrics['f1']:.4f}")
    print(f"ROC-AUC:  {test_metrics['auc']:.4f}")

    print("\n=== CLASSIFICATION REPORT ===")
    print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], digits=4, zero_division=0))


if __name__ == "__main__":
    main()

if you want to see the repartition of the data : 

In [ ]:
loader = LIDDSLoader(path_data)
rows = loader.load_all()

print("Nb samples:", len(rows))
print(rows[0]["file"])
print(rows[0]["split"])
print(rows[0]["scenario"])
print(rows[0]["label"])
print(rows[0]["trace_df"].head())

from collections import Counter

def show_split_stats(rows):
    for split in ["train", "validation", "test"]:
        labels = [r["label"] for r in rows if r["split"] == split]
        print(split, Counter(labels))

show_split_stats(rows)

Nb samples: 1177
/Users/tristan/Documents/dev/01_Cours_CS/08_MLNS/intrusion-detection/data/LIS-DS/Bruteforce_CWE-307/Bruteforce_CWE-307/test/normal/abundant_cohen_3429.zip
test
Bruteforce_CWE-307
0
             timestamp procname      syscall direction  res raw_rest
0  1631064318497514940  apache2        wait4         >  NaN         
1  1631064318497537222  apache2       select         >  NaN         
2  1631064318589388421  apache2       accept         >  NaN  flags=0
3  1631064318589398550  apache2        semop         >  NaN  semid=0
4  1631064318589405667  apache2  getsockname         >  NaN         
